# drive2sim run $run_id

Notebook do `d2s kaggle push` sinh ra. Chạy từ trên xuống; kết quả ghi vào `/kaggle/working/runs/` và `/kaggle/working/cache/`, tải về bằng `d2s kaggle pull $run_id`.

In [ ]:
# Môi trường của phiên Kaggle: ghi lại để điền câu hỏi mở về phiên bản PyTorch.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch

print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
%%bash
# Lấy đúng commit đã tạo run. %%bash có set -e: lỗi thì notebook dừng và báo lỗi, không chạy tiếp
# các ô sau với code sai (lệnh "!" của IPython bỏ qua mã lỗi).
set -euo pipefail
git clone --quiet --recurse-submodules --branch $branch $repo_url /kaggle/tmp/repo
cd /kaggle/tmp/repo
git checkout --quiet $commit
git submodule update --quiet --init --recursive

In [ ]:
%%bash
# Cài môi trường chính bằng đúng script dùng trên máy cá nhân (envs/install_main.sh), để run trên
# Kaggle và trên máy dùng cùng phiên bản PyTorch, gsplat-geer, nvdiffrast, pytorch3d. Build cho
# đúng GPU của phiên này (compute capability đọc từ nvidia-smi, ví dụ 7.5 cho T4, 8.9 cho L4);
# mất khoảng vài chục phút mỗi phiên. Python của Kaggle (3.13 từ 11/2026) mới hơn bản requirements.txt
# hỗ trợ (torch 2.4.1, numpy 1.23.5), nên cài vào một venv Python 3.10 như envs/main.yml và Colab.
set -euo pipefail
cd /kaggle/tmp/repo
command -v uv >/dev/null || python -m pip install --quiet uv
uv venv --quiet --seed --python 3.10 /kaggle/tmp/venv
[ -d /usr/local/cuda/bin ] && export PATH="/usr/local/cuda/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | sort -u | paste -sd';')"
echo "TORCH_CUDA_ARCH_LIST=$TORCH_CUDA_ARCH_LIST"
# nvcc của image khác major với PyTorch thì install_main.sh cài CUDA 12.8 vào đây (~3 GB).
export D2S_CUDA_PREFIX=/kaggle/tmp/cuda-12.8
bash envs/install_main.sh --python /kaggle/tmp/venv/bin/python

In [ ]:
# Gói job (CONTRACTS mục 9.2).
import json
import pathlib

bundle = json.loads(r"""$bundle_json""")
bundle_dir = pathlib.Path("/kaggle/tmp/bundle")
bundle_dir.mkdir(parents=True, exist_ok=True)
for name, content in bundle.items():
    (bundle_dir / name).write_text(json.dumps(content, indent=2))

In [ ]:
%%bash
# Chạy run, rồi luôn dọn output kể cả khi run lỗi, để lần đẩy sau chạy tiếp được từ output này:
# output chỉ giữ runs/ và cache/, bỏ data/ (symlink tới dataset) và cảnh ảo bằng symlink.
# Mã lỗi của run được trả lại sau khi dọn, nên notebook báo "error" khi run lỗi.
set -uo pipefail
cd /kaggle/tmp/repo
/kaggle/tmp/venv/bin/d2s kaggle run /kaggle/tmp/bundle --workdir /kaggle/working --input-root /kaggle/input
rc=$?
rm -rf /kaggle/working/data
find /kaggle/working/runs /kaggle/working/cache -type l -delete 2>/dev/null || true
exit $rc